# Create a Simple Reflex-Based Lunar Lander Agent

In this example, we will use Gymnasium, an environment to train agents via reinforcement learning (RL). We will not use RL here but just learn how the environment works by creating a custom simple reflex-based agent that chooses
actions purely based on the percepts. 

You need Gymnasium installed. Follow the steps in [Setup_Gymnasium.ipynb](../common/Setup_Gymnasium.ipynb).

Below is the installation needed for Colab to run.

In [ ]:
%pip install swig
%pip install pyvirtualdisplay
%pip install gymnasium[box2d,other]

## The Lunar Lander Environment 

![Luna Lander image](https://gymnasium.farama.org/_images/lunar_lander.gif)

The documentation of the environment is available at: https://gymnasium.farama.org/environments/box2d/lunar_lander/

* Performance Measure: The Lunar Lander environment specifies some rewards for during the episode, but we will only 
consider the final reward of -100 or +100 points for crashing or landing safely respectively. This reward structure directly represent the goal of landing safely.

* Environment: This environment is a classic rocket trajectory optimization problem. The space is **continuous** with
  x and y coordinates in the range [-2.5, 2.5]. The landing pad is at coordinate (0,0).

* Actuators: According to Pontryagin’s
  maximum principle, it is optimal to fire the engine at full throttle or turn it off. This is the reason why this environment has discrete actions: engine on or off. There are four discrete actions available:

    - 0: do nothing
    - 1: fire left orientation engine
    - 2: fire main engine
    - 3: fire right orientation engine

* Sensors: Each observation is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

The different ranges/settings of the environment can be queried.

In [23]:
import gymnasium as gym
import numpy as np
np.set_printoptions(precision=2)

In [24]:
def query_environment(name):
    env = gym.make(name)
    print(f"Action Space: {env.action_space}")
    print(f"Observation Space: {env.observation_space}")
    print(f"Max Episode Steps: {env.spec.max_episode_steps}")
    print(f"Nondeterministic: {env.spec.nondeterministic}")
   # print(f"Reward Range: {env.reward_range}")
    print(f"Reward Threshold: {env.spec.reward_threshold}")
    env.close()

query_environment("LunarLander-v3")

Action Space: Discrete(4)
Observation Space: Box([ -2.5   -2.5  -10.   -10.    -6.28 -10.    -0.    -0.  ], [ 2.5   2.5  10.   10.    6.28 10.    1.    1.  ], (8,), float32)
Max Episode Steps: 1000
Nondeterministic: False
Reward Threshold: 200



Gymnasium environments are implemented as classes with a `make` method to create the environment, a `reset` method, and a `step` method to execute an action.
To use it with an agent function that expects percepts and returns an action, we need write glue code that connects the environment with the agent function.

In [25]:
def run_episode(agent_function, env, max_steps=1000, verbose = True, render = True):
    """Run one episode in the environment using the provided agent."""

    # Reset the environment to generate the first observation (use seed=42 in reset to get reproducible results)
    observation, info = env.reset()

    Return = 0
    # run one episode
    for i in range(max_steps):
        # call the agent function to select an action
        action = agent_function(observation)

        # step: execute an action in the environment
        observation_p, reward, terminated, truncated, info = env.step(action)

        if verbose:
            print (f"Step {i+1}: Obs {np.round(observation, 1)} -> Action {action} - > Reward {np.round(reward,1)}, Obs' {np.round(observation_p,1)}")

        observation = observation_p
        Return += reward

        # render the environment
        if render:
            env.render()

        if terminated:
            break
  
    success = reward == 100.0
  
    if verbose:
        print(f"Episode Return: {Return} (success: {success})")
    
    return Return, success

## Example: A Random Agent

We randomly return one of the actions. The environment accepts the integers 0-3.


In [26]:
def random_agent_function(observation): 
    """A random agent that selects actions uniformly at random. It ignores the observation."""
    return np.random.choice([0, 1, 2, 3], p=[0.25, 0.25, 0.25, 0.25])

Run an episode.

In [27]:
env = gym.make("LunarLander-v3", render_mode="human")

run_episode(random_agent_function, env)

env.close()

Step 1: Obs [ 0.   1.4  0.3 -0.6 -0.  -0.1  0.   0. ] -> Action 0 - > Reward -1.3, Obs' [ 0.   1.4  0.3 -0.6 -0.  -0.1  0.   0. ]
Step 2: Obs [ 0.   1.4  0.3 -0.6 -0.  -0.1  0.   0. ] -> Action 3 - > Reward -1.9, Obs' [ 0.   1.4  0.3 -0.6 -0.  -0.1  0.   0. ]
Step 3: Obs [ 0.   1.4  0.3 -0.6 -0.  -0.1  0.   0. ] -> Action 3 - > Reward -2.0, Obs' [ 0.   1.4  0.3 -0.6 -0.  -0.1  0.   0. ]
Step 4: Obs [ 0.   1.4  0.3 -0.6 -0.  -0.1  0.   0. ] -> Action 1 - > Reward -0.9, Obs' [ 0.   1.3  0.3 -0.7 -0.  -0.1  0.   0. ]
Step 5: Obs [ 0.   1.3  0.3 -0.7 -0.  -0.1  0.   0. ] -> Action 2 - > Reward 4.0, Obs' [ 0.   1.3  0.3 -0.6 -0.  -0.1  0.   0. ]
Step 6: Obs [ 0.   1.3  0.3 -0.6 -0.  -0.1  0.   0. ] -> Action 2 - > Reward 1.2, Obs' [ 0.   1.3  0.3 -0.6 -0.  -0.1  0.   0. ]
Step 7: Obs [ 0.   1.3  0.3 -0.6 -0.  -0.1  0.   0. ] -> Action 1 - > Reward -1.0, Obs' [ 0.   1.3  0.3 -0.7 -0.  -0.1  0.   0. ]
Step 8: Obs [ 0.   1.3  0.3 -0.7 -0.  -0.1  0.   0. ] -> Action 1 - > Reward -0.8, Obs' [ 0.

Gymnasium displays environments using `render()` method on the local display. Headless installations like Google Colab do not have a display, but the output can be captured using a virtual display as a video and then add the video to the notebook.

We need to start a virtual display.

Now we can use wrappers for the environment to record the display. The functions are provided in
[display_record.py].



In [28]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("gymnasium_display_recorder.py", 
         "https://raw.githubusercontent.com/mhahsler/Introduction_to_Reinforcement_Learning/refs/heads/main/common/")

In [29]:
from gymnasium_display_recorder import VideoWrapper, show

env = gym.make('LunarLander-v3', render_mode="rgb_array")
env_record = VideoWrapper(env, 'LL1', render_fps=30)

run_episode(random_agent_function, env_record, verbose=False)

show(env_record)

Videos already exist, I remove them first!


/home/mhahsler/github/Introduction_to_Reinforcement_Learning/.venv/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /home/mhahsler/github/Introduction_to_Reinforcement_Learning/Intro/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Showing: ./videos/video_LL1-episode-0.mp4


## A Simple Reflex-Based Agent

To make the code easier to read, we use enumerations for actions (integers) and observations (index in the observation vector).

In [30]:
from enum import Enum

class Act(Enum):
    LEFT = 1
    RIGHT = 3
    MAIN = 2
    NO_OP = 0

class Obs(Enum):
    X = 0
    Y = 1
    VX = 2
    VY = 3
    ANGLE = 4
    ANGULAR_VELOCITY = 5
    LEFT_LEG_CONTACT = 6
    RIGHT_LEG_CONTACT = 7


Define a simple agent that uses the main thruster to reduce the falling speed if it gets too fast.

In [44]:
def rocket_agent_function(observation):
    """A simple agent function."""

    # run the main thruster, if the lander is falling too fast
    if observation[Obs.VY.value] < -.3:  
        return Act.MAIN.value

    return Act.NO_OP.value 

Look at 3 simulation runs.

In [40]:
for _ in range(3):
    Return, success = run_episode(rocket_agent_function, env_record, verbose = False)
    print(f"Episode return {Return} (success: {success})")
    show(env_record)

Episode return -318.51654239397635 (success: False)
Showing: ./videos/video_LL1-episode-4.mp4


Episode return -282.57662077602265 (success: False)
Showing: ./videos/video_LL1-episode-5.mp4


Episode return -395.84122644804495 (success: False)
Showing: ./videos/video_LL1-episode-6.mp4


## Evaluating the Agent

Run the agent on 100 problems and report the average reward.

In [45]:
def run_episodes(agent_function, env, n=100):
    """Run multiple episodes with the given agent and return the rewards for each episode."""
    
    Returns = []
    successes = []
    for _ in range(n):
        Return, success = run_episode(agent_function, env, verbose=False, render=False)
        Returns.append(Return)
        successes.append(success)
        
    return Returns, successes

Run experiments.

In [46]:
Returns, successes = run_episodes(rocket_agent_function, env)
print(successes)

print(f"Average Returns: {np.average(Returns)}")
print(f"Success rate: {np.sum(successes)}/{len(successes)}")

[False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False]
Average Returns: -274.344367690625
Success rate: 2/100


This is not great performance! I am sure we could implement a better reflex-based agent!

This course is about how we can learn an agent's behavior instead of hard-coding it.


&copy; 2025 [Michael Hahsler](http://michael.hahsler.net). 
This work is openly licensed under [Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0) License](https://creativecommons.org/licenses/by-sa/4.0/)

![CC BY-SA 4.0](https://licensebuttons.net/l/by-sa/3.0/88x31.png)